# Day 7 Lab 10: Gold KPIs and Aggregations

        **Layer:** Gold  
        **Python reference:** `Lab_Files/labs/lab_10_gold_kpis_and_aggregations.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Read enriched Silver orders.
- Build daily, category, and segment Gold KPI tables.
- Display each business-ready output separately.

**Dependency:** Run Lab/Notebook 09 first so `lake/silver/orders_enriched` exists.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
LAB_FILES = HERE / "Lab_Files"
if not LAB_FILES.exists():
    LAB_FILES = HERE.parent / "Lab_Files"

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")

## 1. Import Lab Helpers

In [ ]:
from day7_common import LAKE_DIR, gold_frames, ensure_output_dirs, require_source_data, spark_session, write_csv_dir

## 2. Start Spark and Read Enriched Orders

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook10GoldKpis")

enriched = spark.read.parquet(str(LAKE_DIR / "silver" / "orders_enriched"))
enriched.select("order_id", "event_date", "status", "region", "loyalty_tier", "product_category", "amount_usd").show(20, truncate=False)

## 3. Build Gold Frames

In [ ]:
frames = gold_frames(enriched)
gold_path = LAKE_DIR / "gold"
print(sorted(frames.keys()))

## 4. Daily Revenue

In [ ]:
frames["daily_revenue"].show(20, truncate=False)

## 5. Category Revenue

In [ ]:
frames["category_revenue"].show(20, truncate=False)

## 6. Segment Revenue

In [ ]:
frames["segment_revenue"].show(20, truncate=False)

## 7. Write Gold Outputs

In [ ]:
for name, frame in frames.items():
    frame.write.mode("overwrite").parquet(str(gold_path / name))
    write_csv_dir(frame, gold_path / f"{name}_notebook_csv")
print(f"Gold tables written: {', '.join(frames.keys())}")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()